In [1]:
# Installation rapide
!pip install polars s3fs boto3 pyarrow pandas openpyxl --quiet


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


# Chargement dans la base concernée

In [2]:
import boto3
import io
import pandas as pd
from io import BytesIO

# Connexion à MinIO
s3_client = boto3.client(
    "s3",
    endpoint_url="http://minio.mon-namespace.svc.cluster.local:80",
    aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
    aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
    verify=False
)

# Paramètres
bucket_name = "admindataanstat"
object_key = "Solde/DONNEES 2015/012015.xlsx"

# Télécharger le fichier Excel depuis S3
response = s3_client.get_object(Bucket=bucket_name, Key=object_key)
bytes_io = io.BytesIO(response['Body'].read())

# Charger le fichier Excel
excel_file = pd.ExcelFile(bytes_io)

# Afficher les feuilles disponibles
print("Feuilles disponibles dans le fichier Excel :")
print(excel_file.sheet_names)

Feuilles disponibles dans le fichier Excel :
['Donnée1-012015', 'Donnée2-012015']


## Suppression de l'entête du fichier

In [3]:
import pandas as pd
from io import BytesIO

# 1. Lire depuis S3
obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)
data_bytes = obj['Body'].read()

# Charger dans un buffer mémoire pour réutilisation
bio = BytesIO(data_bytes)

# 2. Lire la 2e feuille sans entête pour identifier la bonne ligne d'entête
data_tmp = pd.read_excel(bio, sheet_name=1, header=None, engine='openpyxl')

# Supprimer la dernière ligne (souvent vide ou non utile)
data_tmp = data_tmp.iloc[:-1].reset_index(drop=True)

# 3. Trouver dynamiquement la ligne contenant "matricule"
header_row = None
for i, row in data_tmp.iterrows():
    row_str = row.astype(str).str.lower().str.strip().tolist()
    if any("matricule" in cell for cell in row_str):  # Comparaison insensible à la casse
        header_row = i
        break

if header_row is None:
    raise ValueError("Aucune entête avec 'matricule' détectée !")

# 4. Relire depuis le début du fichier avec la bonne ligne d'entête
bio.seek(0)
data = pd.read_excel(bio, sheet_name=1, header=header_row, engine='openpyxl')

In [4]:
data.head()

,MATRICULE||CODE_ORGANISME,124,125,126,171,190,195,200,201,202,...,491,492,494,495,496,497,498,510,780,BRUT MONTANT
0,011242X15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,34560
1,012856Q15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42400
2,013454N15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45600
3,027861L25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,800000
4,047690HBQ,513605.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,842646


In [5]:
print (data.columns)
print(f"Nombre de lignes : {len(data)}")

Index(['MATRICULE||CODE_ORGANISME', '124', '125', '126', '171', '190', '195',
       '200', '201', '202',
       ...
       '491', '492', '494', '495', '496', '497', '498', '510', '780',
       'BRUT MONTANT'],
      dtype='object', length=147)
Nombre de lignes : 184577


# Fusion (On va segmenter pour faire la fusion de deux années)

## Fusion 2015 et 2016

In [3]:
# 1. Connexion au serveur
bucket_name = "admindataanstat"
prefixes = ["Solde/DONNEES 2015/", "Solde/DONNEES 2016/"]

In [4]:
# 2. Récupérer tous les fichiers Excel au format MMYYYY.xlsx ==========
import re

fichiers_par_mois = {}

for prefix in prefixes:
    response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

    if "Contents" in response:
        for obj in response["Contents"]:
            key = obj["Key"]
            match = re.search(r"(\d{6})\.xlsx$", key)  # Ex : 012015
            if match:
                mois = match.group(1)
                fichiers_par_mois.setdefault(mois, []).append(key)

print(f"🔍 Fichiers détectés pour {len(fichiers_par_mois)} périodes.")

nb_mois = len(fichiers_par_mois)
if nb_mois == 24:
    print("✅ Tous les 24 mois ont bien été détectés.")
else:
    print(f"⚠️ Attention : seulement {nb_mois} mois détectés (au lieu de 24).")
    print("🗂️ Mois détectés :", sorted(fichiers_par_mois.keys()))


🔍 Fichiers détectés pour 24 périodes.
✅ Tous les 24 mois ont bien été détectés.


In [5]:
import pandas as pd
import boto3
from io import BytesIO

# Connexion S3
s3 = boto3.client(
    "s3",
    aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
    aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
    endpoint_url="http://minio.mon-namespace.svc.cluster.local:80",
    verify=False
)

bucket_name = "admindataanstat"
prefix = "Solde/DONNEES "

# Lister tous les fichiers Excel 2015 et 2016
response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

all_files = response.get("Contents", [])
files_2015_2016 = [
    obj["Key"]
    for obj in all_files
    if obj["Key"].endswith(".xlsx") and ("2015" in obj["Key"] or "2016" in obj["Key"])
]

files_2015_2016 = sorted(files_2015_2016)

print(f"Nombre total de fichiers 2015-2016 : {len(files_2015_2016)}")

def lire_fichier_s3_feuille2(key):
    obj = s3.get_object(Bucket=bucket_name, Key=key)
    bytes_io = BytesIO(obj['Body'].read())

    df_raw = pd.read_excel(bytes_io, sheet_name=1, header=None, dtype=str)

    montant_brut_variants = ["montant brut", "brut montant", "total général"]

    print(f"Analyse des lignes dans {key} :")
    header_row = None
    last_non_empty_col = None

    for i, row in df_raw.iterrows():
        row_str = row.astype(str).str.strip()
        row_str_lower = row_str.str.lower()

        if any("matricule" in cell for cell in row_str_lower) and any(
           any(v in cell for v in montant_brut_variants) for cell in row_str_lower):
            for idx, val in enumerate(row_str_lower):
                if any(v in val for v in montant_brut_variants):
                    last_non_empty_col = idx
                    break
            header_row = i
            print(f"--> En-tête trouvée à la ligne {header_row}, colonne {last_non_empty_col}")
            break

    if header_row is None:
        raise ValueError(f"Ligne d'en-tête contenant 'MATRICULE' et 'MONTANT BRUT' introuvable dans {key}")

    bytes_io.seek(0)
    df = pd.read_excel(bytes_io, sheet_name=1, header=header_row, usecols=range(last_non_empty_col + 1), dtype=str)

    last_row = df.tail(1)
    contient_montant_brut = last_row.astype(str).apply(
        lambda x: x.str.lower().str.contains("montant brut|brut montant|total général", regex=True, na=False)
    ).any(axis=1).iloc[0]
    contient_matricule = last_row.astype(str).apply(
        lambda x: x.str.lower().str.contains("matricule", na=False)
    ).any(axis=1).iloc[0]

    if contient_montant_brut and not contient_matricule:
        df = df.iloc[:-1]

    df['source_fichier'] = key.split('/')[-1]
    return df

# Lecture et fusion des fichiers
colonnes_totales = set()
dfs = []

for file_key in files_2015_2016:
    try:
        df = lire_fichier_s3_feuille2(file_key)
        colonnes_totales.update(df.columns)
        dfs.append(df)
    except ValueError as e:
        print(f"⚠️ Ignoré fichier {file_key} : {e}")

colonnes_totales = sorted(colonnes_totales)
dfs_harmonises = [df.reindex(columns=colonnes_totales) for df in dfs]
df_concat = pd.concat(dfs_harmonises, ignore_index=True)

print(f"Nombre total de lignes fusionnées : {len(df_concat)}")
print("Colonnes finales :", colonnes_totales)

# Sauvegarde Parquet sur S3
buffer = BytesIO()
df_concat.to_parquet(buffer, index=False)
buffer.seek(0)

output_key = "Solde/FUSION/fusion_2015_2016.parquet"
s3.put_object(Bucket=bucket_name, Key=output_key, Body=buffer.getvalue())

print(f"✅ Fusion sauvegardée sur S3 : {output_key}")


Nombre total de fichiers 2015-2016 : 24
Analyse des lignes dans Solde/DONNEES 2015/012015.xlsx :
--> En-tête trouvée à la ligne 1, colonne 146
Analyse des lignes dans Solde/DONNEES 2015/022015.xlsx :
--> En-tête trouvée à la ligne 1, colonne 143
Analyse des lignes dans Solde/DONNEES 2015/032015.xlsx :
--> En-tête trouvée à la ligne 1, colonne 145
Analyse des lignes dans Solde/DONNEES 2015/042015.xlsx :
--> En-tête trouvée à la ligne 1, colonne 143
Analyse des lignes dans Solde/DONNEES 2015/052015.xlsx :
--> En-tête trouvée à la ligne 1, colonne 141
Analyse des lignes dans Solde/DONNEES 2015/062015.xlsx :
--> En-tête trouvée à la ligne 1, colonne 147
Analyse des lignes dans Solde/DONNEES 2015/072015.xlsx :
--> En-tête trouvée à la ligne 1, colonne 143
Analyse des lignes dans Solde/DONNEES 2015/082015.xlsx :
--> En-tête trouvée à la ligne 1, colonne 142
Analyse des lignes dans Solde/DONNEES 2015/092015.xlsx :
--> En-tête trouvée à la ligne 1, colonne 142
Analyse des lignes dans Solde/DON

## Fusion 2017 et 2018 

In [6]:
# 1. Connexion au serveur
bucket_name = "admindataanstat"
prefixes = ["Solde/DONNEES 2017/", "Solde/DONNEES 2018/"]

In [7]:
# 2. Récupérer tous les fichiers Excel au format MMYYYY.xlsx ==========
import re

fichiers_par_mois = {}

for prefix in prefixes:
    response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

    if "Contents" in response:
        for obj in response["Contents"]:
            key = obj["Key"]
            match = re.search(r"(\d{6})\.xlsx$", key)  # Ex : 012015
            if match:
                mois = match.group(1)
                fichiers_par_mois.setdefault(mois, []).append(key)

print(f"🔍 Fichiers détectés pour {len(fichiers_par_mois)} périodes.")
nb_mois = len(fichiers_par_mois)
if nb_mois == 24:
    print("✅ Tous les 24 mois ont bien été détectés.")
else:
    print(f"⚠️ Attention : seulement {nb_mois} mois détectés (au lieu de 24).")
    print("🗂️ Mois détectés :", sorted(fichiers_par_mois.keys()))


🔍 Fichiers détectés pour 24 périodes.
✅ Tous les 24 mois ont bien été détectés.


In [ ]:
import pandas as pd
import boto3
from io import BytesIO

# Connexion S3
s3 = boto3.client(
    "s3",
    aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
    aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
    endpoint_url="http://minio.mon-namespace.svc.cluster.local:80",
    verify=False
)

bucket_name = "admindataanstat"
prefix = "Solde/DONNEES "

# Lister tous les fichiers Excel 2017 et 2018
response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

all_files = response.get("Contents", [])
files_2017_2018 = [
    obj["Key"]
    for obj in all_files
    if obj["Key"].endswith(".xlsx") and ("2017" in obj["Key"] or "2018" in obj["Key"])
]

files_2017_2018 = sorted(files_2017_2018)

print(f"Nombre total de fichiers 2017-2018 : {len(files_2017_2018)}")

def lire_fichier_s3_feuille2(key):
    obj = s3.get_object(Bucket=bucket_name, Key=key)
    bytes_io = BytesIO(obj['Body'].read())

    df_raw = pd.read_excel(bytes_io, sheet_name=1, header=None, dtype=str)

    montant_brut_variants = ["montant brut", "brut montant", "total général"]

    print(f"Analyse des lignes dans {key} :")
    header_row = None
    last_non_empty_col = None

    for i, row in df_raw.iterrows():
        row_str = row.astype(str).str.strip()
        row_str_lower = row_str.str.lower()

        if any("matricule" in cell for cell in row_str_lower) and any(
           any(v in cell for v in montant_brut_variants) for cell in row_str_lower):
            for idx, val in enumerate(row_str_lower):
                if any(v in val for v in montant_brut_variants):
                    last_non_empty_col = idx
                    break
            header_row = i
            print(f"--> En-tête trouvée à la ligne {header_row}, colonne {last_non_empty_col}")
            break

    if header_row is None:
        raise ValueError(f"Ligne d'en-tête contenant 'MATRICULE' et 'MONTANT BRUT' introuvable dans {key}")

    bytes_io.seek(0)
    df = pd.read_excel(bytes_io, sheet_name=1, header=header_row, usecols=range(last_non_empty_col + 1), dtype=str)

    last_row = df.tail(1)
    contient_montant_brut = last_row.astype(str).apply(
        lambda x: x.str.lower().str.contains("montant brut|brut montant|total général", regex=True, na=False)
    ).any(axis=1).iloc[0]
    contient_matricule = last_row.astype(str).apply(
        lambda x: x.str.lower().str.contains("matricule", na=False)
    ).any(axis=1).iloc[0]

    if contient_montant_brut and not contient_matricule:
        df = df.iloc[:-1]

    df['source_fichier'] = key.split('/')[-1]
    return df

# Lecture et fusion des fichiers
colonnes_totales = set()
dfs = []

for file_key in files_2017_2018:
    try:
        df = lire_fichier_s3_feuille2(file_key)
        colonnes_totales.update(df.columns)
        dfs.append(df)
    except ValueError as e:
        print(f"⚠️ Ignoré fichier {file_key} : {e}")

colonnes_totales = sorted(colonnes_totales)
dfs_harmonises = [df.reindex(columns=colonnes_totales) for df in dfs]
df_concat = pd.concat(dfs_harmonises, ignore_index=True)

print(f"Nombre total de lignes fusionnées : {len(df_concat)}")
print("Colonnes finales :", colonnes_totales)

# Sauvegarde Parquet sur S3
buffer = BytesIO()
df_concat.to_parquet(buffer, index=False)
buffer.seek(0)

output_key = "Solde/FUSION/fusion_2017_2018.parquet"
s3.put_object(Bucket=bucket_name, Key=output_key, Body=buffer.getvalue())

print(f"✅ Fusion sauvegardée sur S3 : {output_key}")


Nombre total de fichiers 2017-2018 : 24


## Fusion 2019 et 2020

In [6]:
# 1. Connexion au serveur
bucket_name = "admindataanstat"
prefixes = ["Solde/DONNEES 2019/", "Solde/DONNEES 2020/"]

In [7]:
# 2. Récupérer tous les fichiers Excel au format MMYYYY.xlsx ==========
import re

fichiers_par_mois = {}

for prefix in prefixes:
    response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

    if "Contents" in response:
        for obj in response["Contents"]:
            key = obj["Key"]
            match = re.search(r"(\d{6})\.xlsx$", key)  # Ex : 012015
            if match:
                mois = match.group(1)
                fichiers_par_mois.setdefault(mois, []).append(key)

print(f"🔍 Fichiers détectés pour {len(fichiers_par_mois)} périodes.")

nb_mois = len(fichiers_par_mois)
if nb_mois == 24:
    print("✅ Tous les 24 mois ont bien été détectés.")
else:
    print(f"⚠️ Attention : seulement {nb_mois} mois détectés (au lieu de 24).")
    print("🗂️ Mois détectés :", sorted(fichiers_par_mois.keys()))


🔍 Fichiers détectés pour 24 périodes.
✅ Tous les 24 mois ont bien été détectés.


In [ ]:
import pandas as pd
import boto3
from io import BytesIO

# Connexion S3
s3 = boto3.client(
    "s3",
    aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
    aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
    endpoint_url="http://minio.mon-namespace.svc.cluster.local:80",
    verify=False
)

bucket_name = "admindataanstat"
prefix = "Solde/DONNEES "

# Lister tous les fichiers Excel 2019 et 2020
response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

all_files = response.get("Contents", [])
files_2019_2020 = [
    obj["Key"]
    for obj in all_files
    if obj["Key"].endswith(".xlsx") and ("2019" in obj["Key"] or "2020" in obj["Key"])
]

files_2019_2020 = sorted(files_2019_2020)

print(f"Nombre total de fichiers 2019-2020 : {len(files_2019_2020)}")

def lire_fichier_s3_feuille2(key):
    obj = s3.get_object(Bucket=bucket_name, Key=key)
    bytes_io = BytesIO(obj['Body'].read())

    df_raw = pd.read_excel(bytes_io, sheet_name=1, header=None, dtype=str)

    montant_brut_variants = ["montant brut", "brut montant", "total général"]

    print(f"Analyse des lignes dans {key} :")
    header_row = None
    last_non_empty_col = None

    for i, row in df_raw.iterrows():
        row_str = row.astype(str).str.strip()
        row_str_lower = row_str.str.lower()

        if any("matricule" in cell for cell in row_str_lower) and any(
           any(v in cell for v in montant_brut_variants) for cell in row_str_lower):
            for idx, val in enumerate(row_str_lower):
                if any(v in val for v in montant_brut_variants):
                    last_non_empty_col = idx
                    break
            header_row = i
            print(f"--> En-tête trouvée à la ligne {header_row}, colonne {last_non_empty_col}")
            break

    if header_row is None:
        raise ValueError(f"Ligne d'en-tête contenant 'MATRICULE' et 'MONTANT BRUT' introuvable dans {key}")

    bytes_io.seek(0)
    df = pd.read_excel(bytes_io, sheet_name=1, header=header_row, usecols=range(last_non_empty_col + 1), dtype=str)

    last_row = df.tail(1)
    contient_montant_brut = last_row.astype(str).apply(
        lambda x: x.str.lower().str.contains("montant brut|brut montant|total général", regex=True, na=False)
    ).any(axis=1).iloc[0]
    contient_matricule = last_row.astype(str).apply(
        lambda x: x.str.lower().str.contains("matricule", na=False)
    ).any(axis=1).iloc[0]

    if contient_montant_brut and not contient_matricule:
        df = df.iloc[:-1]

    df['source_fichier'] = key.split('/')[-1]
    return df

# Lecture et fusion des fichiers
colonnes_totales = set()
dfs = []

for file_key in files_2019_2020:
    try:
        df = lire_fichier_s3_feuille2(file_key)
        colonnes_totales.update(df.columns)
        dfs.append(df)
    except ValueError as e:
        print(f"⚠️ Ignoré fichier {file_key} : {e}")

colonnes_totales = sorted(colonnes_totales)
dfs_harmonises = [df.reindex(columns=colonnes_totales) for df in dfs]
df_concat = pd.concat(dfs_harmonises, ignore_index=True)

print(f"Nombre total de lignes fusionnées : {len(df_concat)}")
print("Colonnes finales :", colonnes_totales)

# Sauvegarde Parquet sur S3
buffer = BytesIO()
df_concat.to_parquet(buffer, index=False)
buffer.seek(0)

output_key = "Solde/FUSION/fusion_2019_2020.parquet"
s3.put_object(Bucket=bucket_name, Key=output_key, Body=buffer.getvalue())

print(f"✅ Fusion sauvegardée sur S3 : {output_key}")


# Fusion globale des différentes sous fusion bi-annuelle

In [2]:
import pandas as pd
import boto3
from io import BytesIO

# Connexion S3
s3 = boto3.client(
    "s3",
    aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
    aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
    endpoint_url="http://minio.mon-namespace.svc.cluster.local:80",
    verify=False
)

bucket_name = "admindataanstat"

# Fonction pour lire un parquet depuis S3
def read_parquet_s3(key):
    obj = s3.get_object(Bucket=bucket_name, Key=key)
    return pd.read_parquet(BytesIO(obj['Body'].read()))

# Lecture depuis S3
df_15_16 = read_parquet_s3("Solde/FUSION/fusion_2015_2016.parquet")
df_17_18 = read_parquet_s3("Solde/FUSION/fusion_2017_2018.parquet")
df_19_20 = read_parquet_s3("Solde/FUSION/fusion_2019_2020.parquet")

# Harmoniser les noms de colonnes (supprimer espaces, uniformiser casse)
def clean_columns(df):
    df.columns = df.columns.str.strip().str.upper()
    return df

df_15_16 = clean_columns(df_15_16)
df_17_18 = clean_columns(df_17_18)
df_19_20 = clean_columns(df_19_20)

# Append tout en conservant toutes les colonnes
df_final = pd.concat([df_15_16, df_17_18, df_19_20], axis=0, join="outer", ignore_index=True)

# Vérifier la présence des colonnes clés
print("Colonnes finales :", df_final.columns.tolist())

# Sauvegarde sur S3
buffer = BytesIO()
df_final.to_parquet(buffer, index=False)
buffer.seek(0)

output_key = "Solde/FUSION/fusion_2015_2020_feuille2.parquet"
s3.put_object(Bucket=bucket_name, Key=output_key, Body=buffer.getvalue())

print(f"✅ Concaténation terminée : {df_final.shape[0]} lignes x {df_final.shape[1]} colonnes")
print(f"📂 Fichier sauvegardé sur S3 : {output_key}")


Colonnes finales : ['124', '125', '126', '171', '190', '195', '200', '201', '202', '203', '204', '207', '208', '209', '210', '211', '213', '214', '216', '217', '218', '221', '222', '223', '225', '226', '228', '229', '230', '231', '232', '234', '235', '236', '237', '238', '239', '240', '242', '243', '249', '251', '253', '257', '258', '259', '261', '262', '263', '264', '268', '269', '271', '272', '274', '275', '276', '278', '280', '281', '286', '287', '293', '294', '295', '296', '297', '299', '300', '310', '311', '312', '313', '320', '321', '322', '323', '330', '335', '336', '351', '352', '353', '368', '369', '380', '381', '382', '401', '402', '403', '404', '406', '410', '411', '412', '415', '416', '417', '418', '419', '420', '421', '422', '423', '424', '425', '426', '428', '429', '430', '433', '434', '435', '436', '437', '438', '440', '441', '442', '443', '444', '445', '447', '448', '449', '450', '451', '452', '453', '454', '455', '456', '457', '458', '461', '462', '463', '464', '465', 